<span style='color:#0066cc'> <span style='font-family:serif'> <font size="15"> **Access Cloud and Aerosol Lidar (Swath) Data from CALIPSO**


`CAL_LID_L2_01kmCLay-Standard-V5-00` is the Cloud-Aerosol Lidar and Infrared Pathfinder Satellite Observation (**CALIPSO**) Lidar Level 2 1 km Cloud Layer data product. This data product was collected using the Cloud-Aerosol Lidar with Orthogonal Polarization (CALIOP) instrument. Within this cloud layer product, generated at a horizontal resolution of 1 km, are two general classes of data: Column Properties (including position data and viewing geometry) and Layer Properties. The cloud layer products consist of a sequence of column descriptors, each associated with a variable number of cloud layer descriptors. The column descriptors specify the temporal and geophysical location of the column of the atmosphere through which a given lidar pulse travels. Also included in the column descriptors are indicators of surface lighting conditions, information about the surface type, and the number of features (e.g., cloud layers) identified within the column. For each feature within a column, a set of layer descriptors is reported. The layer descriptors provide information about the spatial and optical characteristics of a feature, such as base and top altitudes, integrated attenuated backscatter, and optical depth. CALIPSO was a partnership between NASA and the French Space Agency, CNES. CALIPSO was launched on April 28, 2006 to study the many roles played by clouds and aerosols in Earth’s climate and weather. It flew in the international A-Train constellation for coincident Earth observations from launch until September 13, 2018, when CALIPSO began lowering its orbit from 705 km to 688 km (428 miles) above the Earth to resume formation flying with CloudSat as part of the “C-Train”. The CALIPSO satellite carried three remote sensing instruments: the Cloud-Aerosol Lidar with Orthogonal Polarization (CALIOP), the Imaging Infrared Radiometer (IIR), and the Wide Field-of-View Camera (WFC). By mutual agreement between NASA and CNES, the CALIPSO science mission concluded on August 1, 2023. **Source:** [NASA Earthdata](https://www.earthdata.nasa.gov/data/catalog/larc-cloud-cal-lid-l2-01kmclay-standard-v5-00-v5-00).


 <span style='color:#ff6666'><font size="5"> **Requirements**
1. <font size="3"> EDL authentication (username/password)
2. <font size="3"> Run Notebook in binder from NASA-tutorial Github repository, or
3. <font size="3"> Get `environment.yml` file and install conda environment to run notebook.


 <span style='color:#ff6666'><font size="5"> **Objectives**
### Subset a remote file

- <font size="3">**a)** By Variables
- <font size="3">**b)** By Spatial selection
- <font size="3">**c)** Daytime-only data

### Subset multiple remote files

- <font size="3"> Stream subset of data into local workstation

 <span style='color:#ff6666'><font size="5"> **References**


<font size="3"> **Getzewich, B. (2025)**. <i>CALIPSO Lidar Level 2 1 km Cloud Layer, V5-00</i> [Data set]. NASA Langley Atmospheric Science Data Center Distributed Active Archive Center. https://doi.org/10.5067/CALIOP/CALIPSO/CAL_LID_L2_01KMCLAY-STANDARD-V5-00

In [ ]:
import xarray as xr
import datetime as dt
import earthaccess
import matplotlib.pyplot as plt
import numpy as np

# import pydap-specific tools
from pydap.client import get_cmr_urls, open_url
from pydap.client import to_netcdf as dap_to_netcdf

# Finding OPeNDAP URLs

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Query opendap urls using NASA's CMR API**

<font size="3"> We query NASA's CMR to identify remote files that intersect the following geographical area (bounding box) covering the following time range

- <font size="3"> -121 < longitude < -115, and 26.5 < latitude < 31
- <font size="3"> 2 years of only spring time data: March 1st to May 31st (2022-2023).

 <font size="3"> Lasly, we are **ONLY** interested in `Daytime` data.



In [ ]:
Calipso_L2_ccid = "C3463063995-LARC_CLOUD" # 
bbox = [-121,26.5,-115,31] # [west, south, east, north]

# 2 years of March data
time_ranges = [[dt.datetime(year, 3, 1), dt.datetime(year, 5, 31)] for year in range(2022, 2024)]

CMR_URLs = []
args = {
    "ccid": Calipso_L2_ccid,
    "bounding_box": bbox,
    "limit": 1000,
}
cmr_urls = [url for time_range in time_ranges for url in get_cmr_urls(**args, time_range=time_range)] # you can increase the limit of results

print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **EDL Authentication via earthaccess and OPeNDAP**

<font size="3.5"> You can authenticate via earthaccess as demonstrated below. You must have a valid EDL account. There are two strategies for authenticating with `earthaccess`:

1. `strategy="interactive"`. This will promt your edl username-password.
2. `strategy="netrc"`. Use this if the notebook is running on an environment where a `.netrc` with your credentials is recoverable. T

<font size="3.5"> Below the default will be `netrc`, assuming the user has executed the notebook **Authenticate.ipynb**. If not, you can change the strategy to `"interactive"`.



In [ ]:
from earthaccess.exceptions import LoginStrategyUnavailable
try:
    auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials
except LoginStrategyUnavailable:
    auth = earthaccess.login(strategy="interactive", persist=True)

# pass Token Authorization to a new Session.
my_session = session=auth.get_session()


<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Accessing Metadata-ONLY with PyDAP**

<font size="3"> We can access <span style='color:#ff6666'>**OPeNDAP**</span>-produced metadata to identify the variables of interest. In particular those associated with latitude and longitude values

<font size="3"> Below need to request the <span style='color:#ff6666'>**DAP4**</span> metadata from the remote server. To specify the protocol, we have 2 options:

**1.** <font size="3"> Replace "https" with "dap4" in the URL. This works when using `Xarray`:
```python
open_url(url.replace("https","dap4"), ...)
```
**2**. <font size="3"> Specify the protocol directly (**does not work with Xarray**)
```python
open_url(url, protocol='dap4', ...)
```

<font size="3"> Below we follow option **2)** above.


In [ ]:
%%time
pyds = open_url(cmr_urls[0], protocol="dap4", session=my_session)
pyds.tree()

In [ ]:
output_path = "data/"

## Download minimal variables to identify spatial subset and daytime data

<font size="3"> Coordinates are have (fully qualifying names):

- <font size="3"> `Latitude`
- <font size="3"> `Longitude`
- <font size="3"> `Day_Night_Flag`


<font size="3"> Before donwloading, we need to idenfity any dimension that is also an array of the dataset

<font size="3"> (There can also be Named dimensions, i.e. dimensions that are only named and that are NOT

<font size="3">associated with any data. We do not need to declare those Variables)


In [ ]:
DIMS = list(set(pyds['Latitude'].dims + pyds['Longitude'].dims + pyds['Day_Night_Flag'].dims))
dims = [dim for dim in DIMS if dim.split("/")[-1] in pyds[("/").join(DIMS[1].split('/')[:-1])].variables()] 
print("Dimensions that are also arrays: ", dims)

### Stream data

In [ ]:
%%time
dap_to_netcdf(cmr_urls, session=my_session, 
              keep_variables = ["/Longitude", "/Day_Night_Flag"], 
              output_path=output_path)

## Inspect all downloaded files

<font size="3"> Here, we further identify the subset of data needed on the remote file, that will return ONLY data within our bounding box, for any possible variable of interest.

In [ ]:
# Get data from Bounding Box
minLon, maxLon = bbox[0], bbox[2]

slices=[]
final_urls = []
for url in cmr_urls:
    filename = output_path+f"{url.split('/')[-1][:-4]}.nc4"
    dt1 = xr.open_datatree(filename).load()
    daytime_flag = dt1['Day_Night_Flag']
    # find index /data_01/longitude
    longitude = dt1['/Longitude']
    mask = (longitude >= minLon) & (longitude <= maxLon)
    idx = np.nonzero(mask.values)[0]
    daytime_flag = dt1['Day_Night_Flag'].isel(Record_Number=slice(idx[0], idx[-1]))==1
    if all(daytime_flag==0):
        final_urls.append(url)
        slices.append({"/Record_Number":(idx[0], idx[-1])})

print(f"\nOnly {len(final_urls)} out of the {len(cmr_urls)} remote files satisfy our Daylight Criteria\n")
print("Sample subsetting slices:")
slices[:4]

## Inspect Visually the slice to subset

In [ ]:
# Subset data
Lon = dt1['Longitude'].isel(Record_Number=slice(idx[0], idx[-1]))

# Generate masked data to visualize only

Lon_masked = xr.full_like(dt1['Longitude'], np.nan)
Lon_masked.loc[dict(
    Record_Number = Lon['Record_Number'] + idx[0]
)] = Lon

# Visualize: Plot subset of data over original data
fig, axes = plt.subplots(figsize=(10,4))
dt1['Longitude'].plot(lw=5, color='k', alpha=0.75);
Lon_masked.plot(lw=10, color="#7f00ff")
axes.set_title(r"Longitude Subset $[^\circ$E]")
plt.show()

In [ ]:
output_path='data/'

## Download all data of interest

<font size="3"> **FIRST:** We need to erase all previously downloaded files, to avoid filename collision


In [ ]:
import os
import glob

fnames = [output_path+f"{fname.split('/')[-1][:-4]}.nc4" for fname in cmr_urls]
for filename in fnames:
    try:
        os.remove(filename)
    except FileNotFoundError:
        print(f"The file '{filename}' is not in there anymore")    

In [ ]:
# Will Download 34 Variables!
keep_variables = [
    '/Lidar_Surface_Detection', # <----- ALL Variables inside Group
    "/Ocean_Derived_Column_Optical_Depth", # < -- All varibles inside Group
    "/Lidar_Data_Altitudes", "/Profile_ID", "/Latitude", "/Longitude", 
    "/Profile_Time", "/Profile_UTC_Time", "/Day_Night_Flag", "/Tropopause_Height", 
    "/Tropopause_Temperature",
]


In [ ]:
%%time
dap_to_netcdf(final_urls, session=my_session,
              keep_variables = keep_variables,
              dim_slices = slices,
              output_path=output_path)

## Inspect a downloaded (local) file

<font size="3"> **NOTE:** File inherits the source filename via the OPeNDAP metadata. We can retrieve the source filename from each URL


In [ ]:
filename = output_path+f"{final_urls[0].split('/')[-1][:-4]}.nc4"
dt1 = xr.open_datatree(filename).load()
dt1
